In [26]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import root_mean_squared_error
import time
import pygwalker as pyg
from datetime import datetime
import os
import csv

In [63]:
# train_df = pd.read_csv('../artifacts/train.csv')
# test_df = pd.read_csv('../artifacts/test.csv')
df = pd.read_csv('../artifacts/raw.csv')

C:\Users\Dhvanish\AppData\Local\Temp\ipykernel_16136\13393110.py:3: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../artifacts/raw.csv')


In [64]:
df.isnull().sum()

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

## Imputing missing values ##

In [65]:
df['CompetitionDistance'].fillna(0,inplace=True)
#train_df.isnull().sum()

In [66]:
df['Date'] = pd.to_datetime(df['Date'])

df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day

df['CompetitionOpen_missing'] = df['CompetitionOpenSinceYear'].isna().astype(int)
df['CompetitionOpenSinceMonth'].fillna(1,inplace=True)
df['CompetitionOpenSinceYear'].fillna(df['Year'].min(),inplace=True)
df['CompetitionOpen'] = 12 * (df['Year'] - df['CompetitionOpenSinceYear']) + (df['Month'] - 
                                                                              df['CompetitionOpenSinceMonth'])
df['CompetitionOpen'] = df['CompetitionOpen'].apply(lambda x: max(x, 0))


In [67]:
df['WeekOfYear'] = df['Date'].dt.isocalendar().week
df['Promo2OpenSinceMonths'] = 12 * (df['Year'] - df['Promo2SinceYear']) + (df['WeekOfYear'] - 
                                                                           df['Promo2SinceWeek']) / 4.0
df['Promo2OpenSinceMonths'] = df['Promo2OpenSinceMonths'].apply(lambda x: max(x, 0) if pd.notnull(x) else 0)
df.loc[df['Promo2'] == 0, 'Promo2OpenSinceMonths'] = 0
month_map = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
             7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df['MonthStr'] = df['Month'].map(month_map)
promo_months = df['PromoInterval'].fillna('').str.split(',')
df['IsPromoMonth'] = [
    1 if m in months else 0 
    for m, months in zip(df['MonthStr'], promo_months)
]


In [68]:
df.drop(['Store','Date','CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear','Promo2SinceWeek','Promo2SinceYear','PromoInterval','MonthStr'], axis=1, inplace=True)

In [69]:
df.sample(10)

,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,Promo2,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,IsPromoMonth
444143,4,0,0,0,0,a,0,a,a,540.0,0,2014,5,29,0,35.0,22,0.00,0
204036,5,6869,705,1,1,0,0,a,a,1400.0,1,2015,1,30,0,31.0,5,58.00,1
735431,2,5182,563,1,1,0,1,d,a,3140.0,1,2013,9,10,0,2.0,37,29.75,0
79711,4,7095,467,1,1,0,0,d,c,8990.0,1,2015,5,21,0,66.0,21,56.50,0
600866,4,8579,809,1,1,0,0,d,c,2870.0,0,2014,1,9,0,16.0,2,0.00,0
775900,1,4487,673,1,0,0,1,a,a,620.0,0,2013,8,5,0,0.0,32,0.00,0
609674,3,0,0,0,0,a,1,d,a,1560.0,1,2014,1,1,0,0.0,1,9.75,0
895196,6,6637,514,1,0,0,0,d,c,9790.0,1,2013,4,20,1,3.0,16,44.25,0
473898,6,7174,650,1,0,0,0,a,a,12770.0,0,2014,5,3,0,163.0,18,0.00,0
508352,3,18378,2224,1,1,0,0,a,a,1790.0,0,2014,4,2,0,35.0,14,0.00,0


In [9]:
walker = pyg.walk(df)

Box(children=(HTML(value='\n<div id="ifr-pyg-00065a6ca8bbdd7btjDo2HyRE0SapO5l" style="height: auto">\n    <hea…

In [70]:
df['StateHoliday'] = np.where((df['StateHoliday'] == '0') | (df['StateHoliday'] == 0),0,1)

In [71]:
df['IsSunday'] = (df['DayOfWeek'] == 7).astype(int)

In [72]:
df['IsStoreType_b'] = (df['StoreType'] == 'b').astype(int)

In [ ]:
#df.drop(columns=['StoreType','DayOfWeek'],inplace=True,axis=1)

In [73]:
df.sample(20)

,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,...,Year,Month,Day,CompetitionOpen_missing,CompetitionOpen,WeekOfYear,Promo2OpenSinceMonths,IsPromoMonth,IsSunday,IsStoreType_b
671532,4,5424,509,1,1,0,0,a,a,4300.0,...,2013,11,7,0,56.0,45,4.50,0,0,0
811249,4,4623,496,1,1,0,1,a,c,38710.0,...,2013,7,4,0,3.0,27,0.00,0,0,0
1001965,1,5364,588,1,0,0,0,d,c,6470.0,...,2013,1,14,0,97.0,3,0.00,0,0,0
904089,5,4487,574,1,1,0,0,a,a,660.0,...,2013,4,12,0,75.0,15,0.00,0,0,0
150711,3,6081,735,1,1,0,0,a,c,19360.0,...,2015,3,18,1,26.0,12,0.00,0,0,0
293402,6,7592,988,1,0,0,0,a,c,200.0,...,2014,11,1,0,19.0,44,19.50,0,0,0
240241,6,9637,979,1,0,0,0,c,c,31830.0,...,2014,12,27,0,57.0,52,0.00,0,0,0
1016018,3,7069,1051,1,0,0,1,a,c,70.0,...,2013,1,2,0,271.0,1,6.75,0,0,0
165815,4,8724,812,1,1,0,0,a,c,7180.0,...,2015,3,5,0,28.0,10,0.00,0,0,0
618900,2,2470,357,1,0,0,1,c,c,740.0,...,2013,12,24,1,11.0,52,9.50,1,0,0


In [74]:
num_col=['Customers','CompetitionDistance','CompetitionOpen','Promo2OpenSinceMonths']
cat_col = ['StoreType','Assortment','Year']

In [75]:
X = df.drop(columns=['Sales'])
y = df['Sales']

In [76]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [77]:
preprocessor = ColumnTransformer([
    ('scl',StandardScaler(),num_col),
    ('ohe',OneHotEncoder(drop='first'),cat_col)    
])

In [78]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [79]:
results_file = 'rmsep_score.csv'
file_exists = os.path.isfile(results_file)

models = {
    "XGBRegressor" : XGBRegressor(tree_method='hist',n_jobs=-1),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=50,max_depth=15,n_jobs=-1),
    'LinearRegression': LinearRegression(),
    'LGBMRegressor': LGBMRegressor(n_jobs=-1)
}

results = []

for model_name,model in models.items():
    y_test_mean = np.mean(y_test)

    training_start = time.perf_counter()
    model.fit(X_train,y_train)
    training_stop = time.perf_counter()
    training_time_taken = training_stop - training_start

    prediction_start = time.perf_counter()
    prediction = model.predict(X_test)
    prediction_stop = time.perf_counter()
    prediction_time_taken = prediction_stop-prediction_start
    rmse = root_mean_squared_error(y_test,prediction)
    rmsep = rmse/y_test_mean

    print(f'{model_name}: {rmsep*100:.2f}%\n')
    print(f'Training time taken for {model_name}: {training_time_taken:.4f}\n')
    print(f'prediction time taken for {model_name}: {prediction_time_taken:.4f}\n')

    results.append({
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model_name': model_name,
        'rmse': rmse,
        'rmsep_percent': rmsep * 100,
        'prediction_time_taken_sec': prediction_time_taken,
        'training_time_taken_sec': training_time_taken
    })

with open(results_file, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['timestamp', 'model_name', 'rmse', 'rmsep_percent', 'prediction_time_taken_sec','training_time_taken_sec'])
    if not file_exists:
        writer.writeheader()
    writer.writerows(results)

XGBRegressor: 13.47%

Training time taken for XGBRegressor: 4.8341

prediction time taken for XGBRegressor: 0.1029

RandomForestRegressor: 13.42%

Training time taken for RandomForestRegressor: 38.1456

prediction time taken for RandomForestRegressor: 0.4188

LinearRegression: 23.75%

Training time taken for LinearRegression: 0.2876

prediction time taken for LinearRegression: 0.0138

LGBMRegressor: 15.94%

Training time taken for LGBMRegressor: 2.2037

prediction time taken for LGBMRegressor: 0.2692

